In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [18]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [6]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=10,
        boost_dict={"question": 1.0, "answer": 4.0, "section": 0.2},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [7]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [8]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])
print(result)

LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None), EasyInputMessage(content='I just found this course — can I still sign up now, or is it too late?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"sign up late enrollment registration course still join now too late"}', call_id='call_Kl8XuIVQhNDXT94YK3LKFaz0', name='search', type='function_call', id='fc_0240af8223d1b637006a4d08d2a074819bb89f565a6be82438', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_Kl8XuIVQhNDXT94YK3LKFaz0', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need 

In [9]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I just found this course — can I still sign up now, or is it too late?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"sign up late enrollment registration course still join now too late"}', call_id='call_Kl8XuIVQhNDXT94YK3LKFaz0', name='search', type='function_call', id='fc_0240af8223d1b637006a4d08d2a074819bb89f565a6be82438', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_Kl8XuIVQhNDXT94YK3LKFaz0',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your pr

In [10]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [11]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"sign up late enrollment registration course still join now too late"}'}]

In [12]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [13]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I just found this course — can I still sign up now, or is it too late?',
 'answer_agent': 'Yes — you can still join the course now.\n\nThe FAQ says: **“Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.”**\n\nAlso, you **don’t need a registration confirmation email** to start learning; you can just begin the materials and submit homework while the submission form is open.\n\nIf you want, I can also tell you what matters for the certificate vs. what you can still do self-paced.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"sign up late enrollment registration course still join now too late"}'}],
 'cost': Decimal('0.00213375'),
 'document': '74eb249bbf'}

In [14]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [19]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [20]:
df_agent = pd.DataFrame(agent_answers)
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,I just found this course — can I still sign up...,Yes — you can still join.\n\nAccording to the ...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""si...",0.002025,74eb249bbf
1,Am I allowed to join the course after it alrea...,Yes — you can still join the course after it h...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""jo...",0.0018810,74eb249bbf
2,"If I enroll late, do I still get access to eve...","Yes — you can start whenever you want, and the...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.0021915,74eb249bbf
3,Can I still get a certificate if I join the co...,"Yes, you can still join now, but to get a cert...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""ce...",0.00174225,74eb249bbf
4,What do I need to do to make sure I’m eligible...,To stay eligible for the certificate if you st...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""ce...",0.00235125,74eb249bbf


In [21]:
df_agent["cost"].sum()

Decimal('0.10408200')

In [22]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [23]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [41]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [40]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [44]:

import json


from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, list):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [45]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It says you can still join, and it correctly adds that to receive a certificate you must submit your project while submissions are still being accepted. That preserves the key point from the original answer.', answer_score='good', trajectory_reasoning='The single search query is relevant to the user’s question about whether it is too late to sign up. While it is somewhat generic and not tailored to the specific course FAQ wording, one search call is reasonable and there are no unnecessary duplicate searches. The tool call supports the final answer.', trajectory_score='good')

In [46]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [47]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [48]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [49]:
df_agent_eval = pd.DataFrame(agent_evaluations)
df_agent_eval.head()

,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,I just found this course — can I still sign up...,74eb249bbf,good,The agent's answer matches the ground truth. I...,good,The tool query was relevant to the question ab...
1,Am I allowed to join the course after it alrea...,74eb249bbf,good,The agent’s answer matches the ground truth. I...,good,The search query is relevant to the user’s que...
2,"If I enroll late, do I still get access to eve...",74eb249bbf,bad,The agent’s answer does not match the ground t...,good,The search query was relevant to the question ...
3,Can I still get a certificate if I join the co...,74eb249bbf,good,The agent’s answer matches the ground truth on...,good,The search query was relevant to the question ...
4,What do I need to do to make sure I’m eligible...,74eb249bbf,good,The agent’s answer matches the ground truth on...,good,The search query was relevant to the question ...


In [50]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    47
bad      3
Name: count, dtype: int64

In [51]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    49
bad      1
Name: count, dtype: int64

In [52]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)